# Induction Evals
We evaluate that the induction case is strong enough to drive a wedge between LLM performance in this notebook.

**Replication design:** each (archetype, info type) condition runs **R = 30 replicates** of the same n=9-harmonic quiz, each under a fresh seed (fresh labels + noise padding, with that seed reused as the request decoding seed). Per the power analysis in `power_analysis.py`, R = 29 achieves ≥80% power for every pairwise harmonic-stratified CMH contrast at Bonferroni α = 0.05/18 and certifies the three near-tie contrasts equivalent within ±0.15 accuracy by TOST at α = 0.05/3; R = 30 adds headroom. Each replicate serializes to `results/{archetype}_{info}/rep_{seed}.yaml` as soon as it is graded (the preliminary single run is `rep_1776.yaml`), so interrupted sections resume where they left off.

In [1]:
"""The following generates the Quiz all our models will be evaluated on."""

import string
import yaml

import logging

from dotenv import load_dotenv
from pathlib import Path

logging.basicConfig(level=logging.INFO)
load_dotenv(Path.cwd() / "keys.env", verbose=True)

from smolbench.induction.periodic import (
    PeriodicConfig,
    Prompter,
    get_periodic_numeric_quiz,
    numeric_count_query_gen,
)
from smolbench.evals import Marks
import os

# Inference runs on a self-provisioned EC2 spot instance serving vLLM's
# OpenAI-compatible API (see smolbench/evals/ec2.py). Provider dispatch
# happens at import time, so this must be set BEFORE importing
# smolbench.evals.provider below.
os.environ["INFERENCE_PROVIDER"] = "ec2"

# Per-archetype model names: keys of EC2_DEPLOY_SPECS in
# smolbench/evals/ec2.py. Each key is also vLLM's --served-model-name, so it
# goes in the OpenAI request body verbatim. All three are FP8 checkpoints
# (uniform precision; the BF16 405B/Maverick would not fit 8xH100 anyway)
# from UNGATED repos -- no HF account/login/token needed: Meta gates its own
# FP8 releases behind the Llama license, so the Llama models come from Red
# Hat AI (vLLM's quantization team) and the CoT model from NVIDIA's own
# official release.
DENSE_MODEL = "llama-31-405b"      # RedHatAI/Meta-Llama-3.1-405B-Instruct-FP8-dynamic (dense)
COT_MODEL = "nemotron-ultra-253b"  # nvidia/Llama-3_1-Nemotron-Ultra-253B-v1-FP8 (dense CoT)
MOE_MODEL = "llama4-maverick"      # RedHatAI/Llama-4-Maverick-17B-128E-Instruct-FP8 (MoE)

from smolbench.evals.provider import evaluate

template = string.Template(
    "You are a precise integer counter.\n"
    "\n"
    "Task: answer the question below with a single integer and nothing else.\n"
    "\n"
    "Output format:\n"
    "Return exactly one integer and nothing else.\n"
    "Do not output any explanation, punctuation, quotes, or extra whitespace.\n"
    "Stop immediately after writing the integer.\n"
    "\n"
    "Context:\n"
    "There is a counting game. Positions are counted starting from 1. "
    "At each position, words are written according to the following rules:\n"
    "$positive_info\n"
    "Question:\n"
    "How many of the positions 1 through $seq_len include '$label'?"
)

# --- Replication setup -----------------------------------------------------
# Power analysis (see power_analysis.py): R=29 replicates per condition powers
# every pairwise harmonic-stratified CMH contrast at >=80% (Bonferroni
# alpha=0.05/18) and certifies the three near-tie contrasts as equivalent
# within +/-0.15 accuracy via TOST (Bonferroni alpha=0.05/3). R=30 adds one
# replicate of headroom.
#
# A replicate is the SAME n=9-harmonic quiz regenerated under a fresh seed:
# fresh random labels and noise padding (PeriodicConfig.seed) plus that same
# seed as the per-request decoding seed. The 9 harmonics are difficulty
# strata, not iid draws -- power comes from replicating per harmonic, never
# from raising n (that changes difficulty and the lcm(1..n) context length).
R: int = 30
BASE_SEED: int = 1776  # seed of the original preliminary run == replicate 0
REPLICATE_SEEDS: tuple[int, ...] = tuple(BASE_SEED + r for r in range(R))
INFO_TYPES: tuple[str, ...] = ("intens", "extens", "noise_intens")


def make_quizzes(seed: int) -> dict[str, tuple]:
    """Generates one replicate's three info-type quizzes, keyed by info type."""
    return dict(
        zip(
            INFO_TYPES,
            get_periodic_numeric_quiz(
                PeriodicConfig(
                    n=9,
                    labels=9,
                    seed=seed,
                ),
                Prompter(
                    template,
                    {},
                    numeric_count_query_gen,
                ),
            ),
        )
    )


REPLICATES: dict[int, dict[str, tuple]] = {
    seed: make_quizzes(seed) for seed in REPLICATE_SEEDS
}

# First-replicate aliases for the Prompt Validation cells below.
intens_quiz = REPLICATES[BASE_SEED]["intens"]
extens_quiz = REPLICATES[BASE_SEED]["extens"]
noise_intens_quiz = REPLICATES[BASE_SEED]["noise_intens"]

In [2]:
# Replicated evaluation harness. Each (archetype, info type, seed) replicate
# is serialized to results/{tag}_{info}/rep_{seed}.yaml IMMEDIATELY after it
# is graded, so a spot interruption or kernel restart loses at most one
# replicate's work; reruns skip already-serialized replicates, which makes
# the archetype cells below idempotent and resumable. The filename carries
# the seed, keeping every generation reproducible/attributable.
ARCHETYPE_TAGS: dict[str, str] = {
    DENSE_MODEL: "decode",
    COT_MODEL: "cot",
    MOE_MODEL: "moe",
}


def run_replicates(model: str, extra_args: dict | None = None) -> None:
    """Runs all outstanding replicates of every info type against model."""
    tag: str = ARCHETYPE_TAGS[model]
    for seed in REPLICATE_SEEDS:
        for info in INFO_TYPES:
            out = Path("results") / f"{tag}_{info}" / f"rep_{seed}.yaml"
            if out.exists():
                continue  # already graded in a previous (partial) run
            marks: Marks = evaluate(
                REPLICATES[seed][info], model, seed, extra_args=extra_args
            )
            out.parent.mkdir(parents=True, exist_ok=True)
            with open(out, "w") as file:
                yaml.dump(marks, file, default_flow_style=False, indent=4)
            logging.info(
                f"{tag}/{info} seed={seed}: {marks.correct}/{len(marks.marks)} correct"
            )


def summarize(model: str) -> None:
    """Prints per-info-type totals over every serialized replicate of model."""
    tag: str = ARCHETYPE_TAGS[model]
    for info in INFO_TYPES:
        correct = incorrect = invalid = 0
        paths = sorted((Path("results") / f"{tag}_{info}").glob("rep_*.yaml"))
        for path in paths:
            with open(path) as file:
                marks: Marks = yaml.unsafe_load(file)
            correct += marks.correct
            incorrect += marks.incorrect
            invalid += marks.invalid
        print(
            f"{tag}/{info}: {len(paths)}/{R} replicates -- "
            f"correct={correct} incorrect={incorrect} invalid={invalid}"
        )


In [4]:
# Provision ONE large EC2 spot instance for the whole experiment (the
# archetype sections below only swap which model its vLLM serves). This cell
# is idempotent: it reattaches via .ec2_state.json / the smolbench:experiment
# tag instead of launching a second box, so it is safe after kernel restarts.
#
# Safety nets if you forget the Teardown cell at the bottom: an on-instance
# watchdog terminates the box after EC2_IDLE_TIMEOUT_MIN (default 30) idle
# minutes, and an absolute EC2_MAX_LIFETIME_MIN (default 24h) backstop exists.
# That also means: provision right before running the sections, not hours
# ahead. Spot price is ~$30-45/h for p5e/p5 -- mind the meter.
from smolbench.evals.ec2 import (
    agent_status,
    provision_spot_instance,
    serve_model,
    shutdown_instance,
)

state = provision_spot_instance()
print(
    f"instance {state['instance_id']} ({state['instance_type']}) "
    f"in {state['availability_zone']} at {state['public_ip']}"
)

INFO:botocore.credentials:Found credentials in shared credentials file: ~/.aws/credentials
INFO:botocore.credentials:Found credentials in shared credentials file: ~/.aws/credentials
INFO:root:control agent up at 52.12.143.102:9000
INFO:root:provision_spot_instance: reattached to i-0032aae9cfc777a89 (p5e.48xlarge @ us-west-2, 52.12.143.102)


instance i-0032aae9cfc777a89 (p5e.48xlarge) in us-west-2c at 52.12.143.102


## Prompt Validation

In [5]:
print(intens_quiz[0].prompt)

You are a precise integer counter.

Task: answer the question below with a single integer and nothing else.

Output format:
Return exactly one integer and nothing else.
Do not output any explanation, punctuation, quotes, or extra whitespace.
Stop immediately after writing the integer.

Context:
There is a counting game. Positions are counted starting from 1. At each position, words are written according to the following rules:
Every 1 positions write jo.
Every 2 positions write my.
Every 3 positions write fp.
Every 4 positions write xx.
Every 5 positions write bp.
Every 6 positions write yj.
Every 7 positions write dh.
Every 8 positions write qx.
Every 9 positions write fe.

Question:
How many of the positions 1 through 2520 include 'jo'?


In [6]:
print(extens_quiz[0].prompt)

You are a precise integer counter.

Task: answer the question below with a single integer and nothing else.

Output format:
Return exactly one integer and nothing else.
Do not output any explanation, punctuation, quotes, or extra whitespace.
Stop immediately after writing the integer.

Context:
There is a counting game. Positions are counted starting from 1. At each position, words are written according to the following rules:
Position 1: jo.
Position 2: jo|my.
Position 3: jo|fp.
Position 4: jo|my|xx.
Position 5: jo|bp.
Position 6: jo|my|fp|yj.
Position 7: jo|dh.
Position 8: jo|my|xx|qx.
Position 9: jo|fp|fe.
Position 10: jo|my|bp.
Position 11: jo.
Position 12: jo|my|fp|xx|yj.
Position 13: jo.
Position 14: jo|my|dh.
Position 15: jo|fp|bp.
Position 16: jo|my|xx|qx.
Position 17: jo.
Position 18: jo|my|fp|yj|fe.
Position 19: jo.
Position 20: jo|my|xx|bp.
Position 21: jo|fp|dh.
Position 22: jo|my.
Position 23: jo.
Position 24: jo|my|fp|xx|yj|qx.
Position 25: jo|bp.
Position 26: jo|my.
Posi

In [7]:
print(noise_intens_quiz[0].prompt)

You are a precise integer counter.

Task: answer the question below with a single integer and nothing else.

Output format:
Return exactly one integer and nothing else.
Do not output any explanation, punctuation, quotes, or extra whitespace.
Stop immediately after writing the integer.

Context:
There is a counting game. Positions are counted starting from 1. At each position, words are written according to the following rules:
Every 1 positions write jo.
Every 2 positions write my.
Every 3 positions write fp.
Every 4 positions write xx.
Every 5 positions write bp.
Every 6 positions write yj.
Every 7 positions write dh.
Every 8 positions write qx.
Every 9 positions write fe.
JTR3kCGrfcri2K3fN5eoCLmbmOQJj4s1PA5UnHSXZxYL VQg6UEi7L o6XfZk146lRI23DfZQOvKVZSwuwzOiW2Aob9INrzKTKWylgTt3dhrT6 FzbTPFi3c5pg yL4Nbm6NJPl

vNSs X9TusXjMlKOVcL VKu2 Nvyisy291lBcVzeGJKGu0cqsQNomwFebXXaL46DHBRoD eI6Q
IfnuWfrf39D72DpPH rNstYeBR5eDWPVCrdT50 1qY0aCp q7 zBzwQgcT
pq14b808WWYd3dcxQSB 4OTH95YCR5RHHboOeU1VqdEg1a

# Decoder-Only Model
This section tests classical decoder-only models.

In [7]:
# Points the shared instance's vLLM at DENSE_MODEL for this section (a
# container swap; the first serve of a model waits on its checkpoint download,
# reruns hit the instance's HF cache). Exit leaves the instance running for
# the next section. Runs all R replicates x 3 info types; safe to rerun after
# an interruption -- finished replicates are skipped.
with serve_model(DENSE_MODEL):
    run_replicates(DENSE_MODEL)

INFO:root:serve_model: requesting 'llama-31-405b' (RedHatAI/Meta-Llama-3.1-405B-Instruct-FP8-dynamic) ...
INFO:root:serve_model: 'llama-31-405b' is up at http://35.89.102.40:8000/v1
INFO:root:serve_model: background S3 cache upload kicked off for 'llama-31-405b'
INFO:root:decode/intens seed=1777: 9/9 correct
INFO:root:decode/extens seed=1777: 1/9 correct
INFO:root:decode/noise_intens seed=1777: 0/9 correct
INFO:root:decode/intens seed=1778: 9/9 correct
INFO:root:decode/extens seed=1778: 1/9 correct
INFO:root:decode/noise_intens seed=1778: 0/9 correct
INFO:root:decode/intens seed=1779: 9/9 correct
INFO:root:decode/extens seed=1779: 1/9 correct
INFO:root:decode/noise_intens seed=1779: 1/9 correct
INFO:root:decode/intens seed=1780: 9/9 correct
INFO:root:decode/extens seed=1780: 2/9 correct
INFO:root:decode/noise_intens seed=1780: 1/9 correct
INFO:root:decode/intens seed=1781: 8/9 correct
INFO:root:decode/extens seed=1781: 1/9 correct
INFO:root:decode/noise_intens seed=1781: 1/9 correct
IN

In [8]:
# Prints aggregate results over all serialized decode replicates.
summarize(DENSE_MODEL)

decode/intens: 30/30 replicates -- correct=262 incorrect=8 invalid=0
decode/extens: 30/30 replicates -- correct=33 incorrect=237 invalid=0
decode/noise_intens: 30/30 replicates -- correct=18 incorrect=250 invalid=2


## CoT Model
This section tests a CoT model.

In [8]:
# Swaps the instance's vLLM to COT_MODEL for this section. Nemotron-Ultra's
# chain-of-thought is toggled by the "detailed thinking on" system prompt,
# which query() injects from its EC2_DEPLOY_SPECS entry -- user prompts stay
# byte-identical across archetypes. The model emits <think>...</think> as
# plain text (its Llama tokenizer has no think token ids, so vLLM's
# server-side reasoning parsers cannot run for it); query() splits that block
# into the reasoning channel client-side, so scoring sees only the answer.
with serve_model(COT_MODEL):
    run_replicates(COT_MODEL, extra_args={"max_completion_tokens": 8192})

INFO:root:serve_model: requesting 'nemotron-ultra-253b' (nvidia/Llama-3_1-Nemotron-Ultra-253B-v1-FP8) ...
INFO:root:serve_model: 'nemotron-ultra-253b' is up at http://52.12.143.102:8000/v1
INFO:root:serve_model: background S3 cache upload kicked off for 'nemotron-ultra-253b'
INFO:root:cot/extens seed=1783: 8/9 correct
INFO:root:cot/noise_intens seed=1783: 6/9 correct
INFO:root:cot/intens seed=1784: 7/9 correct
INFO:root:cot/extens seed=1784: 9/9 correct
INFO:root:cot/noise_intens seed=1784: 5/9 correct
INFO:root:cot/intens seed=1785: 7/9 correct
INFO:root:cot/extens seed=1785: 7/9 correct
INFO:root:EC2 endpoint request failed on attempt 1: HTTPConnectionPool(host='52.12.143.102', port=8000): Read timed out. (read timeout=120). Retrying in 60 seconds.
INFO:root:cot/noise_intens seed=1785: 2/9 correct
INFO:root:cot/intens seed=1786: 6/9 correct
INFO:root:cot/extens seed=1786: 6/9 correct
INFO:root:cot/noise_intens seed=1786: 8/9 correct
INFO:root:cot/intens seed=1787: 5/9 correct
INFO:ro

In [9]:
# Prints aggregate results over all serialized cot replicates.
summarize(COT_MODEL)

cot/intens: 30/30 replicates -- correct=230 incorrect=40 invalid=0
cot/extens: 30/30 replicates -- correct=219 incorrect=51 invalid=0
cot/noise_intens: 30/30 replicates -- correct=178 incorrect=92 invalid=0


## MoE Model
This section tests an MoE model.

In [10]:
# Swaps the instance's vLLM to MOE_MODEL for this section.
with serve_model(MOE_MODEL):
    run_replicates(MOE_MODEL)

INFO:root:serve_model: requesting 'llama4-maverick' (RedHatAI/Llama-4-Maverick-17B-128E-Instruct-FP8) ...
INFO:root:serve_model: 'llama4-maverick' is up at http://52.12.143.102:8000/v1
INFO:root:serve_model: background S3 cache upload kicked off for 'llama4-maverick'
INFO:root:moe/intens seed=1777: 8/9 correct
INFO:root:moe/extens seed=1777: 3/9 correct
INFO:root:moe/noise_intens seed=1777: 5/9 correct
INFO:root:moe/intens seed=1778: 8/9 correct
INFO:root:moe/extens seed=1778: 3/9 correct
INFO:root:moe/noise_intens seed=1778: 4/9 correct
INFO:root:moe/intens seed=1779: 9/9 correct
INFO:root:moe/extens seed=1779: 2/9 correct
INFO:root:moe/noise_intens seed=1779: 4/9 correct
INFO:root:moe/intens seed=1780: 8/9 correct
INFO:root:moe/extens seed=1780: 2/9 correct
INFO:root:moe/noise_intens seed=1780: 2/9 correct
INFO:root:moe/intens seed=1781: 7/9 correct
INFO:root:moe/extens seed=1781: 3/9 correct
INFO:root:moe/noise_intens seed=1781: 5/9 correct
INFO:root:moe/intens seed=1782: 9/9 correc

In [11]:
# Prints aggregate results over all serialized moe replicates.
summarize(MOE_MODEL)

moe/intens: 30/30 replicates -- correct=236 incorrect=34 invalid=0
moe/extens: 30/30 replicates -- correct=73 incorrect=197 invalid=0
moe/noise_intens: 30/30 replicates -- correct=131 incorrect=139 invalid=0


# Teardown
Gracefully shuts the experiment's EC2 spot instance down. If this cell is forgotten, the instance still self-terminates: an on-instance watchdog fires after `EC2_IDLE_TIMEOUT_MIN` (default 30) minutes without inference traffic or control-agent activity, and an absolute `EC2_MAX_LIFETIME_MIN` (default 24h) `shutdown -h` backstop is scheduled at boot. Both run on the instance itself, so they work even if this notebook's kernel is gone.

In [ ]:
# Terminates the spot instance (and its EBS volume) and clears
# .ec2_state.json. Also works after a kernel restart or a lost state file: it
# falls back to the smolbench:experiment instance tag.
shutdown_instance()

# HRM Model
This section tests an HRM model.

In [ ]:
# hrm_intens_eval: Marks = evaluate(intens_quiz, "perplexity/pplx-embed-v1-4b", SEED)

In [ ]:
# hrm_extens_eval: Marks = evaluate(extens_quiz, "perplexity/pplx-embed-v1-4b", SEED)

In [ ]:
# print(hrm_intens_eval.correct, hrm_intens_eval.incorrect, hrm_intens_eval.invalid)
# print(hrm_extens_eval.correct, hrm_extens_eval.incorrect, hrm_extens_eval.invalid)
# with open('results/hrm_intens.yaml', 'w') as file:
#     yaml.dump(hrm_intens_eval, file, default_flow_style=False, indent=4)
# with open('results/hrm_extens.yaml', 'w') as file:
#     yaml.dump(hrm_extens_eval, file, default_flow_style=False, indent=4)